# 3.2.4 花朵智能识别系统交互流程设计

## 任务说明（摘自考试预览）

花朵智能识别系统在现代城市绿化管理中起着越来越重要的作用，其利用先进的计算机视觉技术，如花朵检测与识别，实现了对花朵种类的实时监控与管理。本系统要求开发一个基于已训练模型的花朵检测与分类系统，能够准确识别出不同类别的花朵。

AI模型说明：提供的模型“flower-detection.onnx”是使用 Pytorch 框架和基于深度卷积神经网络训练得到的，专门用于进行花朵识别。对应的标签文件为“labels.txt”。 该模型的使用交互流程为：

1)加载模型“flower-detection.onnx”和加载类别标签“labels.txt”；

2)加载一张本地花朵图片“flower_test.png”，并预处理图像；

3)使用flower-detection模型对花朵图片进行识别；

4)输出花朵的预测类型和识别的准确率。

你作为一名人工智能训练师，请完成以下工作任务：

（1）补全该模型的使用交互流程对应的Python代码（3.2.4.ipynb），实现本地测试图片“flower _test.png”的识别，将其识别结果截图保存为jpg格式文件，命名为3.2.4-1.jpg。

（2）在上面的使用交互流程基础上，给出在花朵智能识别系统中使用“flower-detection.onnx”模型的一种人机交互的最优流程，将其保存为docx文件，命名为3.2.4.docx。

所有结果文件储存在桌面新建的考生文件夹中，文件夹命名为“准考证号+身份证号后六位”。

---

以下为**刷题参考模板**：含完整可运行代码（编程题）或答题示例与生成 docx 草稿代码（主观题）。

交互页面设计：
图片资源上传
模型选择与预加载
推理结果展示
帮助文档与支持
反馈模块

## 花朵识别（单类别概率）

In [ ]:
import onnxruntime as ort
import numpy as np
import scipy.special
from PIL import Image


def preprocess_image(
    image,
    resize_size=256,
    crop_size=224,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
):
    image = image.resize((resize_size, resize_size), Image.BILINEAR)
    w, h = image.size
    left = (w - crop_size) / 2
    top = (h - crop_size) / 2
    image = image.crop((left, top, left + crop_size, top + crop_size))
    image = np.array(image).astype(np.float32)
    image = image / 255.0
    image = (image - mean) / std
    image = np.transpose(image, (2, 0, 1))
    image = image.reshape((1,) + image.shape)
    return image


session = ort.InferenceSession(
    'ResNet50.onnx', providers=['CPUExecutionProvider']
)

with open('labels.txt', encoding='utf-8') as f:
    labels = [line.strip() for line in f.readlines()]

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

image = Image.open('demo.jpg').convert('RGB')
processed_image = preprocess_image(image).astype(np.float32)

output = session.run(
    [output_name], {input_name: processed_image}
)[0]

accuracy = scipy.special.softmax(output, axis=-1)[0]
predicted_idx = int(np.argmax(accuracy))
prob_percentage = float(accuracy[predicted_idx]) * 100
predicted_label = labels[predicted_idx]

print(
    f"Predicted class: {predicted_label}, Accuracy: {prob_percentage:.2f}%"
)

## 学习提示

- 编程题：请在**本案例文件夹**下运行 Notebook，以便相对路径 `data/...` 正确。
- 主观题：示例答案仅供参考，可按考场要求改写；若未安装 `python-docx`，可先 `pip install python-docx`。
- 交卷前核对平台要求的截图、HTML、docx 命名与保存路径。